# Faruq-v3 retained models — Adrian external validation
Evaluasi post-hoc tanpa training pada validation Adrian nyata: D0FT, SAF1, IGEM1, ACMC1, AF2, dan GEO1. Test tidak diekstrak.

In [ ]:
from google.colab import drive
drive.mount('/content/drive',force_remount=True)
import os, shutil, subprocess, sys, time
from pathlib import Path
REPO=Path('/content/coffee-bean-detection'); BRANCH='agent/af2-igem-paired-confirmation'
if REPO.exists(): shutil.rmtree(REPO)
for attempt in range(3):
    result=subprocess.run(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)])
    if result.returncode == 0: break
    if REPO.exists(): shutil.rmtree(REPO)
    if attempt == 2: raise RuntimeError('Git clone gagal tiga kali')
    time.sleep(2)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO)],check=True)
sys.path.insert(0,str(REPO/'src')); os.chdir(REPO)

In [ ]:
import torch
from coffee_detector.archive_sni21_pilot import restore_real_a0_validation
from coffee_detector.drive_project import resolve_drive_project_root,require_project_artifact
from coffee_detector.experiments.run_faruq_v3_acmc_adrian_external import prepare_adrian_external_validation
assert torch.cuda.is_available(),'Aktifkan T4 GPU'
REQUIRED=('bundles/sni21-vadcp-pilot-bundle/A0_real.tar','evidence/faruq-grouped-development-v1/faruq_grouped_manifest.json')
PROJECT_ROOT=resolve_drive_project_root(required_relative_paths=REQUIRED)
A0=restore_real_a0_validation(require_project_artifact(PROJECT_ROOT,REQUIRED[0]),'/content/sni21-a0-adrian-external-source')
assert not (A0/'test').exists()
ADRIAN_ROOT=Path('/content/sni21-adrian-retained-external-val')
setup=prepare_adrian_external_validation(A0,require_project_artifact(PROJECT_ROOT,REQUIRED[1]),ADRIAN_ROOT)
ADRIAN_SETUP=ADRIAN_ROOT/'adrian_external_validation_summary.json'
OUTPUT_ROOT=PROJECT_ROOT/'experiments/faruq-v3-retained-adrian-external-v1'
print('GPU:',torch.cuda.get_device_name(0),'| images:',setup['images'],'| parents:',setup['independent_parent_ids'])

In [ ]:
MODELS=['D0FT','SAF1','IGEM1','ACMC1','AF2','GEO1']
command=[sys.executable,'-u','-m','coffee_detector.experiments.run_faruq_v3_retained_adrian_external','--project-root',str(PROJECT_ROOT),'--data-root',str(ADRIAN_ROOT),'--adrian-setup',str(ADRIAN_SETUP),'--output-root',str(OUTPUT_ROOT),'--models',*MODELS,'--device','0']
LOG=OUTPUT_ROOT/'run.log'; OUTPUT_ROOT.mkdir(parents=True,exist_ok=True)
process=subprocess.Popen(command,cwd=REPO,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
with LOG.open('a',encoding='utf-8') as log:
    for line in process.stdout:
        print(line,end='',flush=True); log.write(line); log.flush()
code=process.wait()
if code != 0:
    print('\n'.join(LOG.read_text(errors='replace').splitlines()[-120:]))
    raise RuntimeError(f'Evaluasi gagal: {code}; log={LOG}')

In [ ]:
import json,pandas as pd
from IPython.display import display
SUMMARY=OUTPUT_ROOT/'retained_adrian_external_summary.json'
result=json.loads(SUMMARY.read_text())
table=pd.DataFrame(result['rows'])
percentage=[c for c in table.columns if 'map50_95' in c or c=='recall']
display(table.style.format({c:'{:+.2%}' if c.startswith('delta_') else '{:.2%}' for c in percentage}))
print('TRAINING:',result['training_executed'],'| TEST:',result['test_images_accessed'])
print('LIMIT:',result['claim_limit'])
print('SUMMARY:',SUMMARY)